# Contract Validation: Strict and Non-Strict Mode

Mockture validates every interaction against the OpenAPI contract in two phases. The first phase runs at configuration time—when `respond()` is called—and checks whether the template's rendered response body matches the declared schema. The second phase runs at request time and checks the incoming request body against the schema for that endpoint.

The `strict` flag controls what happens when a violation is detected at runtime. In strict mode, a violating request is answered with HTTP 500 and the test fails immediately. In non-strict mode, the configured response is served normally while the violation is silently recorded for later inspection. Both modes accumulate violations and expose them through `assert_no_contract_violations()`.

By completing this notebook you will have observed config-time validation raise `ContractConfigError` in strict mode, seen the same invalid template accepted silently in non-strict mode, compared the HTTP 500 and 201 responses produced by a violating request in each mode, and confirmed that `assert_no_contract_violations()` surfaces accumulated violations in both modes.

## Table of Contents

- [Prerequisites](#Prerequisites)
- [1. Setup](#1-Setup)
- [2. Phase 1 — Config-Time Validation](#2-Phase-1--Config-Time-Validation)
  - [2.1 Schema Mismatch with strict=True](#21-Schema-Mismatch-with-strictTrue)
  - [2.2 Schema Mismatch with strict=False](#22-Schema-Mismatch-with-strictFalse)
- [3. Phase 2 — Runtime Validation](#3-Phase-2--Runtime-Validation)
  - [3.1 A Violating Request in Strict Mode](#31-A-Violating-Request-in-Strict-Mode)
  - [3.2 A Violating Request in Non-Strict Mode](#32-A-Violating-Request-in-Non-Strict-Mode)
- [4. Violation Accumulation](#4-Violation-Accumulation)
  - [4.1 Multiple Violations Recorded](#41-Multiple-Violations-Recorded)
- [5. A Valid Request](#5-A-Valid-Request)
  - [5.1 No Violations on a Well-Formed Request](#51-No-Violations-on-a-Well-Formed-Request)
- [6. Conclusion](#6-Conclusion)

## Prerequisites

No environment variables are required.

Additional prerequisites:

- `mockture` must be installed in the current Python environment.
- `httpx` must be installed (`pip install httpx`).
- `configs/basic_api.openapi.yml` and `configs/basic_api.templates.yml` must be present in the same directory as this notebook. The templates file includes an `invalid_success_shape` template that intentionally violates the response schema.

## 1. Setup

We import `Mockture` and `ContractConfigError`, then define a helper that constructs a fresh instance for each example with either `strict=True` or `strict=False`.

In [ ]:
from pathlib import Path
import httpx
from mockture.server import Mockture
from mockture.errors import ContractConfigError

ROOT      = Path(".").resolve()
CONTRACT  = str(ROOT / "configs" / "basic_api.openapi.yml")
TEMPLATES = str(ROOT / "configs" / "basic_api.templates.yml")

def new_mock(strict):
    return Mockture(contract_path=CONTRACT, templates_path=TEMPLATES, strict=strict)

## 2. Phase 1 — Config-Time Validation

When `respond()` is called, Mockture renders the template's response body and validates it against the OpenAPI response schema for that method, path, and status code. This check runs before the server starts and before any request arrives.

The `invalid_success_shape` template in `configs/basic_api.templates.yml` is intentionally broken: it declares `{"bad_field": "should_fail"}` as a 201 body, while the `OrderResponse` schema requires `order_id` and `status`.

- **ContractConfigError**: raised when a template's rendered response body does not conform to the OpenAPI response schema for that method, path, and status code.

### 2.1 Schema Mismatch with strict=True

With `strict=True`, `respond()` raises `ContractConfigError` immediately when the rendered body does not match the schema. The server is never constructed.

In [ ]:
mock = new_mock(strict=True)

try:
    mock.respond("invalid_success_shape")
    print("No error raised (unexpected)")
except ContractConfigError as e:
    print("strict=True: ContractConfigError raised at respond() time:")
    print(" ", e)

### 2.2 Schema Mismatch with strict=False

With `strict=False`, `respond()` accepts the invalid template without raising. The violation is recorded internally and can be surfaced later with `assert_no_contract_violations()`.

In [ ]:
mock = new_mock(strict=False)

mock.respond("invalid_success_shape")
print("strict=False: respond() completed without raising.")
print("The violation is recorded internally.")

## 3. Phase 2 — Runtime Validation

When a request arrives, Mockture validates the request body against the OpenAPI request schema for that endpoint. The `CreateOrderRequest` schema requires `item_id` with `minLength: 1` and `quantity` with `minimum: 1`. Sending `{"item_id": "", "quantity": 0}` violates both constraints.

The two sections below send the same violating request in strict and non-strict mode and compare the responses.

### 3.1 A Violating Request in Strict Mode

In strict mode, the server responds with HTTP 500 and a `{"error": "contract_violation"}` body instead of the configured response. The violation is also recorded.

In [ ]:
mock_strict = new_mock(strict=True)
mock_strict.respond("create_order_success")
mock_strict.start()

r = httpx.post(
    mock_strict.url_for("/orders"),
    json={"item_id": "", "quantity": 0},  # violates minLength and minimum
    timeout=5.0,
)

print("strict=True — violating request:")
print("  Status:", r.status_code)  # 500
print("  Body:  ", r.json())

mock_strict.stop()

### 3.2 A Violating Request in Non-Strict Mode

In non-strict mode, the same violating request receives the configured 201 response. The violation is recorded but does not interrupt the response.

In [ ]:
mock_ns = new_mock(strict=False)
mock_ns.respond("create_order_success", order_id="ord-ns")
mock_ns.start()

r = httpx.post(
    mock_ns.url_for("/orders"),
    json={"item_id": "", "quantity": 0},  # same violation as above
    timeout=5.0,
)

print("strict=False — same violating request:")
print("  Status:", r.status_code)  # 201 — configured response returned
print("  Body:  ", r.json())

mock_ns.stop()

## 4. Violation Accumulation

Violations are accumulated across the lifetime of a server instance in both modes. `assert_no_contract_violations()` raises `AssertionError` listing all recorded violations. This allows a test to send multiple requests and verify compliance once at the end.

### 4.1 Multiple Violations Recorded

We send two violating requests in non-strict mode, then call `assert_no_contract_violations()` to confirm that both were recorded.

In [ ]:
mock = new_mock(strict=False)
mock.respond("create_order_success", order_id="ord-acc")
mock.start()

httpx.post(mock.url_for("/orders"), json={"item_id": "",  "quantity": 0},    timeout=5.0)
httpx.post(mock.url_for("/orders"), json={"item_id": "x", "quantity": 9999}, timeout=5.0)

try:
    mock.assert_no_contract_violations()
    print("No violations (unexpected)")
except AssertionError as e:
    print("assert_no_contract_violations() failed — violations recorded:")
    print(str(e)[:400])  # trimmed for readability

mock.stop()

## 5. A Valid Request

A request that satisfies all schema constraints passes both validation phases without recording any violations.

### 5.1 No Violations on a Well-Formed Request

We send a request with a valid `item_id` (non-empty string) and a valid `quantity` (between 1 and 100), then confirm that `assert_no_contract_violations()` passes.

In [ ]:
mock = new_mock(strict=True)
mock.respond("create_order_success", order_id="ord-valid")
mock.start()

r = httpx.post(
    mock.url_for("/orders"),
    json={"item_id": "SKU-1", "quantity": 2},  # valid against CreateOrderRequest
    timeout=5.0,
)

print("Status:", r.status_code)
mock.assert_no_contract_violations()
print("No violations — request and response both match the schema.")

mock.stop()

## 6. Conclusion

This notebook covered both phases of Mockture's contract validation:

- Observed `ContractConfigError` raised at `respond()` time with `strict=True` when a template body does not match the response schema.
- Confirmed that the same invalid template is accepted silently with `strict=False`, with the violation recorded internally.
- Compared the HTTP 500 response (strict mode) and the configured 201 response (non-strict mode) produced by the same schema-violating request.
- Verified that `assert_no_contract_violations()` surfaces accumulated violations in non-strict mode.
- Confirmed that a well-formed request produces no violations in either mode.

Use `strict=True` (the default) when you want violations to fail the test immediately at the point they occur. Use `strict=False` when testing client behavior that is independent of schema compliance, such as verifying how the client parses a response body regardless of whether the request that triggered it was valid.